# **Import**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from data_loader import Dataset
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import optuna

# **Explore Techniques to Evaluate and Select Models**


### **Load and Split (Train, Valid, Test)**

In [2]:
dataset = Dataset("binary classification")
X_train, X_test, y_train, y_test = dataset.load_split_data()

X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.25, random_state=1)

### **Train and Validate**

In [3]:
models = {
    "Logistic Regression (None)": LogisticRegression(penalty=None, solver="lbfgs", max_iter=5000),
    "Logistic Regression (L2)": LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000),
    "Logistic Regression (L1)": LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000),
    "Logistic Regression (ElasticNet)": LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=5000),
    "Naïve Bayes": GaussianNB(),
    "XGBoost": XGBClassifier()
}

for name, model in models.items():

  model.fit(X_train, y_train)

  y_pred = model.predict(X_valid)

  accuracy = accuracy_score(y_valid, y_pred)

  print(f"{name} Accuracy: {accuracy:.3f}")

Logistic Regression (None) Accuracy: 0.964
Logistic Regression (L2) Accuracy: 0.971
Logistic Regression (L1) Accuracy: 0.971
Logistic Regression (ElasticNet) Accuracy: 0.971
Naïve Bayes Accuracy: 0.934
XGBoost Accuracy: 0.956


### **K-Fold Cross-Validate**


In [4]:
X_train, X_test, y_train, y_test = dataset.load_split_data()

for name, model in models.items():

    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")

    print(f"{name} Mean Accuracy: {np.mean(scores):.3f}")

Logistic Regression (None) Mean Accuracy: 0.962
Logistic Regression (L2) Mean Accuracy: 0.960
Logistic Regression (L1) Mean Accuracy: 0.960
Logistic Regression (ElasticNet) Mean Accuracy: 0.960
Naïve Bayes Mean Accuracy: 0.952
XGBoost Mean Accuracy: 0.960


### **Leverage Pipeline to Prevent Data Leakage During Standardization**

In [5]:
model_elastic = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=5000)

pipeline_elastic = make_pipeline(StandardScaler(), model_elastic)

scores_elastic = cross_val_score(pipeline_elastic, X_train, y_train, cv=5, scoring="accuracy")

print(f"Mean Accuracy: {np.mean(scores_elastic):.3f}")

Mean Accuracy: 0.960


### **Tune Logistic Regression with Grid Search**

In [6]:
param_grid = {
    "C": [0.01, 0.1, 0.5, 1, 5, 10, 15, 20],
    "penalty": ["l1", "l2"],
}

def objective(trial):

    C = trial.suggest_categorical("C", param_grid["C"])
    penalty = trial.suggest_categorical("penalty", param_grid["penalty"])

    model = LogisticRegression(C=C, penalty=penalty, solver="liblinear", max_iter=5000)

    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")

    return np.mean(scores)

sampler_grid = optuna.samplers.GridSampler(param_grid)

study_logistic = optuna.create_study(direction="maximize", sampler=sampler_grid)

study_logistic.optimize(objective)

print(f"Grid Search best hyperparameters: {study_logistic.best_params}")
print(f"Grid Search best performance: {study_logistic.best_value:.3f}")

[I 2026-04-28 12:32:51,331] A new study created in memory with name: no-name-c43b84f4-e773-4994-80ff-e5d70b500699
[I 2026-04-28 12:32:51,358] Trial 0 finished with value: 0.9230692243536278 and parameters: {'C': 0.01, 'penalty': 'l2'}. Best is trial 0 with value: 0.9230692243536278.
[I 2026-04-28 12:32:51,376] Trial 1 finished with value: 0.9597331109257714 and parameters: {'C': 1, 'penalty': 'l1'}. Best is trial 1 with value: 0.9597331109257714.
[I 2026-04-28 12:32:51,428] Trial 2 finished with value: 0.9597331109257714 and parameters: {'C': 5, 'penalty': 'l1'}. Best is trial 1 with value: 0.9597331109257714.
[I 2026-04-28 12:32:51,449] Trial 3 finished with value: 0.9597331109257714 and parameters: {'C': 5, 'penalty': 'l2'}. Best is trial 1 with value: 0.9597331109257714.
[I 2026-04-28 12:32:51,497] Trial 4 finished with value: 0.9597331109257714 and parameters: {'C': 15, 'penalty': 'l2'}. Best is trial 1 with value: 0.9597331109257714.
[I 2026-04-28 12:32:51,510] Trial 5 finished wi

[I 2026-04-28 12:32:51,571] Trial 6 finished with value: 0.9560800667222686 and parameters: {'C': 0.1, 'penalty': 'l1'}. Best is trial 1 with value: 0.9597331109257714.
[I 2026-04-28 12:32:51,598] Trial 7 finished with value: 0.9615679733110924 and parameters: {'C': 20, 'penalty': 'l1'}. Best is trial 7 with value: 0.9615679733110924.
[I 2026-04-28 12:32:51,667] Trial 8 finished with value: 0.9615679733110924 and parameters: {'C': 10, 'penalty': 'l1'}. Best is trial 7 with value: 0.9615679733110924.
[I 2026-04-28 12:32:51,728] Trial 9 finished with value: 0.9597164303586322 and parameters: {'C': 1, 'penalty': 'l2'}. Best is trial 7 with value: 0.9615679733110924.
[I 2026-04-28 12:32:51,746] Trial 10 finished with value: 0.9597331109257714 and parameters: {'C': 20, 'penalty': 'l2'}. Best is trial 7 with value: 0.9615679733110924.
[I 2026-04-28 12:32:51,757] Trial 11 finished with value: 0.9597331109257714 and parameters: {'C': 10, 'penalty': 'l2'}. Best is trial 7 with value: 0.96156797

Grid Search best hyperparameters: {'C': 20, 'penalty': 'l1'}
Grid Search best performance: 0.962


### **Tune Logistic Regression with Random Search**

In [7]:
sampler_random = optuna.samplers.RandomSampler()
study_logistic = optuna.create_study(direction="maximize", sampler=sampler_random)
study_logistic.optimize(objective, n_trials=10)

print(f"Random Search best hyperparameters: {study_logistic.best_params}")
print(f"Random Search best performance: {study_logistic.best_value:.3f}")

[I 2026-04-28 12:32:51,899] A new study created in memory with name: no-name-ba1031df-caa6-4ff0-8877-d67bf76597c2
[I 2026-04-28 12:32:51,912] Trial 0 finished with value: 0.9230692243536278 and parameters: {'C': 0.01, 'penalty': 'l2'}. Best is trial 0 with value: 0.9230692243536278.
[I 2026-04-28 12:32:51,923] Trial 1 finished with value: 0.9230692243536278 and parameters: {'C': 0.01, 'penalty': 'l2'}. Best is trial 0 with value: 0.9230692243536278.
[I 2026-04-28 12:32:52,013] Trial 2 finished with value: 0.9615679733110924 and parameters: {'C': 10, 'penalty': 'l1'}. Best is trial 2 with value: 0.9615679733110924.
[I 2026-04-28 12:32:52,033] Trial 3 finished with value: 0.9615679733110924 and parameters: {'C': 10, 'penalty': 'l1'}. Best is trial 2 with value: 0.9615679733110924.
[I 2026-04-28 12:32:52,092] Trial 4 finished with value: 0.9597331109257714 and parameters: {'C': 0.5, 'penalty': 'l1'}. Best is trial 2 with value: 0.9615679733110924.
[I 2026-04-28 12:32:52,140] Trial 5 finis

Random Search best hyperparameters: {'C': 10, 'penalty': 'l1'}
Random Search best performance: 0.962


### **Tune Logistic Regression with Bayesian Search**

In [8]:
study_logistic = optuna.create_study(direction="maximize")
study_logistic.optimize(objective, n_trials=10)

print(f"Bayesian Search best hyperparameters: {study_logistic.best_params}")
print(f"Bayesian Search best performance: {study_logistic.best_value:.3f}")

[I 2026-04-28 12:32:52,245] A new study created in memory with name: no-name-c5ed2db9-0937-48a9-aae5-6f2a311e1f08
[I 2026-04-28 12:32:52,262] Trial 0 finished with value: 0.9597331109257714 and parameters: {'C': 20, 'penalty': 'l2'}. Best is trial 0 with value: 0.9597331109257714.
[I 2026-04-28 12:32:52,364] Trial 1 finished with value: 0.9597331109257714 and parameters: {'C': 10, 'penalty': 'l2'}. Best is trial 0 with value: 0.9597331109257714.
[I 2026-04-28 12:32:52,378] Trial 2 finished with value: 0.9597331109257714 and parameters: {'C': 20, 'penalty': 'l2'}. Best is trial 0 with value: 0.9597331109257714.
[I 2026-04-28 12:32:52,427] Trial 3 finished with value: 0.9615679733110924 and parameters: {'C': 15, 'penalty': 'l1'}. Best is trial 3 with value: 0.9615679733110924.
[I 2026-04-28 12:32:52,462] Trial 4 finished with value: 0.9615679733110924 and parameters: {'C': 10, 'penalty': 'l1'}. Best is trial 3 with value: 0.9615679733110924.
[I 2026-04-28 12:32:52,477] Trial 5 finished w

Bayesian Search best hyperparameters: {'C': 15, 'penalty': 'l1'}
Bayesian Search best performance: 0.962


### **Tune XGBoost**

In [9]:
param_grid = {
    "n_estimators": [50, 350],

    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1],

    "learning_rate": [0.01, 0.3],
    "reg_lambda": [0.01, 5],
    "gamma": [0.01, 5],

    "max_depth": [3, 7],
    "min_child_weight": [0.5, 10]
}

def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", param_grid["n_estimators"][0], param_grid["n_estimators"][1])

    subsample = trial.suggest_float("subsample", param_grid["subsample"][0], param_grid["subsample"][1])
    colsample_bytree = trial.suggest_float("colsample_bytree", param_grid["colsample_bytree"][0], param_grid["colsample_bytree"][1])

    learning_rate = trial.suggest_float("learning_rate", param_grid["learning_rate"][0], param_grid["learning_rate"][1])
    reg_lambda = trial.suggest_float("reg_lambda", param_grid["reg_lambda"][0], param_grid["reg_lambda"][1])
    gamma = trial.suggest_float("gamma", param_grid["gamma"][0], param_grid["gamma"][1])

    max_depth = trial.suggest_int("max_depth", param_grid["max_depth"][0], param_grid["max_depth"][1])
    min_child_weight = trial.suggest_float("min_child_weight", param_grid["min_child_weight"][0], param_grid["min_child_weight"][1])

    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_lambda=reg_lambda,
        gamma=gamma,
        min_child_weight=min_child_weight,
    )

    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")

    return np.mean(scores)

In [10]:
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective, n_trials=50)

print(f"Bayesian Search best hyperparameters: {study_xgb.best_params}")
print(f"Bayesian Search best performance: {study_xgb.best_value:.3f}")

[I 2026-04-28 12:32:52,640] A new study created in memory with name: no-name-0b58248d-055e-4c62-82a3-6c4b12f9a5b6
[I 2026-04-28 12:33:02,012] Trial 0 finished with value: 0.9578815679733111 and parameters: {'n_estimators': 62, 'subsample': 0.8221066557676742, 'colsample_bytree': 0.8531870469943527, 'learning_rate': 0.2400196223872147, 'reg_lambda': 0.43099710893296683, 'gamma': 4.5096252953018885, 'max_depth': 3, 'min_child_weight': 1.2030212195085772}. Best is trial 0 with value: 0.9578815679733111.
[I 2026-04-28 12:33:31,683] Trial 1 finished with value: 0.9670558798999165 and parameters: {'n_estimators': 324, 'subsample': 0.8917763628808729, 'colsample_bytree': 0.8838198361931643, 'learning_rate': 0.07803500848785852, 'reg_lambda': 2.9156227947551505, 'gamma': 3.807232869249364, 'max_depth': 5, 'min_child_weight': 4.490996211708446}. Best is trial 1 with value: 0.9670558798999165.
[I 2026-04-28 12:33:49,419] Trial 2 finished with value: 0.9652210175145954 and parameters: {'n_estimat

Bayesian Search best hyperparameters: {'n_estimators': 293, 'subsample': 0.8217840051510331, 'colsample_bytree': 0.9420609438401631, 'learning_rate': 0.0230032122072592, 'reg_lambda': 4.035688049054995, 'gamma': 2.7012558480700664, 'max_depth': 7, 'min_child_weight': 6.817399072233658}
Bayesian Search best performance: 0.971


### **Evaluate XGBoost**

In [11]:
model = XGBClassifier(**study_xgb.best_params)
model.fit(X_train, y_train)

y_pred_raw = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

type_1 = np.sum((y_pred == 1) & (y_test == 0)) / np.sum(y_test == 0)
type_2 = np.sum((y_pred == 0) & (y_test == 1)) / np.sum(y_test == 1)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

print(f"Type I: {type_1:.3f}")
print(f"Type II: {type_2:.3f}")

print(f"Recall: {recall_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"F1: {f1_score(y_test, y_pred):.3f}")

print(f"AUC: {roc_auc_score(y_test, y_pred_raw):.3f}")

Accuracy: 0.993
Type I: 0.000
Type II: 0.021
Recall: 0.979
Precision: 1.000
F1: 0.989
AUC: 1.000
